In [ ]:
import os
import json
import joblib
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from typing import Dict, Any, Optional, Tuple, List

# ---------- Helper: load latest curves cache for a class ----------
def load_curves_cache_for_class(class_name: str, cache_dir: str, key: Optional[str] = None) -> Dict[str, Any]:
    def _load_npz(path: str) -> Dict[str, np.ndarray]:
        with np.load(path, allow_pickle=False) as f:
            return {k: f[k] for k in f.files}
    def _load_json(path: str) -> dict:
        with open(path, "r", encoding="utf-8") as fh:
            return json.load(fh)

    cache_path = os.path.join(cache_dir, class_name)
    files = [] if not os.path.isdir(cache_path) else sorted(
        [f for f in os.listdir(cache_path) if f.endswith(".npz") and "-curves-" in f]
    )
    if key:
        npz_path = os.path.join(cache_path, f"{key}.npz")
        meta_path = os.path.join(cache_path, f"{key}.json")
    else:
        if not files:
            raise FileNotFoundError(f"No cached curves for class '{class_name}'.")
        fname = files[-1]
        npz_path = os.path.join(cache_path, fname)
        meta_path = npz_path.replace(".npz", ".json")

    out = _load_npz(npz_path)
    try:
        out["meta"] = _load_json(meta_path)
    except Exception:
        out["meta"] = {}
    return out


# ========== 1) Loader: prepare everything for one class ==========
def prepare_deeproc_class(
    class_name: str,
    fig_dir: str,
    cache_dir: str
) -> Dict[str, Any]:
    """
    Lädt für class_name:
      - DeepROC-Objekt (dra_loaded)
      - Gruppen-Metriken (measure_list)
      - gecachte curves (curves)
    Gibt ein Bundle-Dict zurück.
    """
    dra_path = os.path.join(fig_dir, class_name, f"DeepROC_{class_name}_object.pkl")
    dra_loaded = joblib.load(dra_path)

    metrics_json = os.path.join(fig_dir, class_name, f"DeepROC_{class_name}_groups.json")
    with open(metrics_json, "r", encoding="utf-8") as fh:
        metrics_data = json.load(fh)
    measure_list = metrics_data["groups"]

    curves = load_curves_cache_for_class(class_name, cache_dir)

    return dict(
        class_name=class_name,
        dra_loaded=dra_loaded,
        measure_list=measure_list,
        curves=curves
    )

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from typing import Dict, Any, Optional, Tuple, List

def _remove_builtin_random_baseline(ax, tol: float = 1e-6) -> None:
    """
    Remove the diagonal 'random baseline' drawn by DeepROC.plotGroup().
    Works whether the line has 2 points or many points.
    """
    for ln in list(ax.lines):
        x = np.asarray(ln.get_xdata(), dtype=float)
        y = np.asarray(ln.get_ydata(), dtype=float)
        if x.size >= 2 and y.size == x.size and np.all(np.isfinite(x)) and np.all(np.isfinite(y)):
            # Is it essentially the diagonal?
            if np.nanmax(np.abs(y - x)) < 1e-6:
                # Does it span (roughly) the unit square?
                if (min(x.min(), y.min()) <= 0 + 5*tol) and (max(x.max(), y.max()) >= 1 - 5*tol):
                    ln.remove()

def _style_lines(ax, lw_roc: float, lw_all: float, other_ls: str = "--", roc_min_pts: int = 20):
    """
    Set linewidths: ROC gets lw_roc, everything else lw_all.
    'ROC' is detected as the line with many vertices (len(x) > roc_min_pts).
    """
    for line in ax.get_lines():
        x = line.get_xdata()
        if len(x) > roc_min_pts:
            line.set_linewidth(lw_roc)
            line.set_linestyle("-")
        else:
            line.set_linewidth(lw_all)
            line.set_linestyle(other_ls)
            line.set_color("black")

def plot_deeproc_from_bundle(
    bundle: Dict[str, Any],
    *,
    tpr_groups: List[Tuple[float, float]],
    costs_dict: Dict[str, float],
    out_dir: str = "figures",
    defaults: Optional[dict] = None,
    fig_size: Tuple[float, float] = (7, 5),
    sens_color: str = "#ffff89",      # horizontal band (sensitivity)
    spec_color: str = "#b2c6f5",      # vertical band (specificity)
    range_auc_color: str = "#99cc99", # overlap (range AUC)
    save_pdf: bool = False,
    show: bool = False
) -> List[str]:
    """
    Render DeepROC plots from a prepared bundle.

    Legend composition:
      - 'Random' dashed baseline (line)
      - Overall AUC (text-only with white bbox)
      - TPR-Range: [lo–hi] (text-only)
      - AUC TPR-Range (Patch with range_auc_color)
      - avgSens TPR-Range (Patch with sens_color)
      - avgSpec TPR-Range (Patch with spec_color)
    """
    class_name   = bundle["class_name"]
    dra_loaded   = bundle["dra_loaded"]
    measure_list = bundle["measure_list"]
    curves       = bundle["curves"]

    # ---- style defaults ----
    DEFAULTS = dict(
        dpi=600,
        transparent=True,
        grid=True,
        tight=True,
        line_width=3, 
        lw_roc=3.0,   
        lw_all=1.2,   
        fonts=dict(title=16, label=26, tick=22, legend=12)
    )
    if defaults:
        DEFAULTS.update({k: v for k, v in defaults.items() if k != "fonts"})
        if "fonts" in defaults and isinstance(defaults["fonts"], dict):
            DEFAULTS["fonts"].update(defaults["fonts"])

    FS_LABELS   = DEFAULTS["fonts"]["label"]
    FS_TICKS    = DEFAULTS["fonts"]["tick"]
    FS_LEGEND   = DEFAULTS["fonts"]["legend"]
    LW_ROC      = DEFAULTS["lw_roc"]
    LW_ALL      = DEFAULTS["lw_all"]
    USE_GRID    = DEFAULTS["grid"]
    TIGHT       = DEFAULTS["tight"]
    DPI         = DEFAULTS["dpi"]
    TRANSPARENT = DEFAULTS["transparent"]
    TICK_LENGTH = 4
    TICK_WIDTH  = 1

    os.makedirs(out_dir, exist_ok=True)
    base_path = os.path.join(out_dir, f"DeepROC_{class_name}")

    # overall AUC (+ CI + prevalence) from cache
    auc_overall = float(curves["auc"][0])
    ci_lo, ci_hi = curves["auc_ci"]
    prev = float(curves["baseline"][0])
    overall_label = f"AUC = {auc_overall:.3f}\n({ci_lo:.3f}, {ci_hi:.3f})\n[{prev*100:.2f}%]"
    # overall_label = f"AUC = {auc_overall:.3f} ({ci_lo:.3f}, {ci_hi:.3f}) [{prev*100:.2f}%]"

    outputs: List[str] = []

    for i, (lo, hi) in enumerate(tpr_groups):
        fig, ax = dra_loaded.plotGroup(
            plotTitle=None,
            groupIndex=i,
            showError=False,
            showThresholds=True,
            showOptimalROCpoints=True,
            costs=costs_dict,
            saveFileName=None,
            numShowThresh=1,
            showPlot=False,
            labelThresh=False,
            full_fpr_tpr=True
        )

        fig.set_size_inches(*fig_size)
        
        roc_line = max(ax.get_lines(), key=lambda ln: len(ln.get_xdata()))
        roc_line.set_label(overall_label)

        # formatting
        fig.patch.set_alpha(0.0)
        fig.set_facecolor("none")
        ax.set_title("")
        ax.set_xlabel("False Positive Rate", fontsize=FS_LABELS)
        ax.set_ylabel("True Positive Rate", fontsize=FS_LABELS)
        ax.tick_params(axis="both", which="major",
                       labelsize=FS_TICKS, length=TICK_LENGTH, width=TICK_WIDTH)
        if USE_GRID:
            ax.grid(True)

        # (1) Entferne evtl. eingebaute Random-Baseline der Library
        _remove_builtin_random_baseline(ax)

        # (2) Style vorhandene Linien: ROC dick, alle anderen dünn/gestrichelt
        _style_lines(ax, lw_roc=LW_ROC, lw_all=LW_ALL, other_ls="--", roc_min_pts=20)

        # (3) Random-Baseline neu und konsistent einzeichnen (bekommt LW_ALL)
        ax.plot([0, 1], [0, 1], linestyle="-.", color="black",
                linewidth=LW_ROC, label="Random")

        # ticks
        ax.set_xticks(np.arange(0, 1.01, 0.2))
        ax.set_yticks(np.arange(0, 1.01, 0.2))

        # metrics
        m     = measure_list[i]
        auc_i = float(m.get("AUCn_i",  np.nan))
        sens  = float(m.get("avgSens", np.nan))
        spec  = float(m.get("avgSpec", np.nan))
        rng   = f"[{lo:.2f} – {hi:.2f}]"

        handles, _labels = ax.get_legend_handles_labels()
        # overall AUC (white background)
        # handles.append(Line2D([], [], linestyle="none", label=overall_label))
        # TPR-Range (Text)
        handles.append(Line2D([], [], linestyle="none", label=f"TPR-Range = {rng}"))
        handles.append(Patch(
            facecolor=range_auc_color, edgecolor='none',
            label=rf"$\mathrm{{AUC}}_{{\text{{TPR-Range}}}} = {auc_i:.3f}$"
        ))
        handles.append(Patch(
            facecolor=sens_color, edgecolor='none',
            label=rf"$\mathrm{{avgSens}}_{{\text{{TPR-Range}}}} = {sens:.3f}$"
        ))
        handles.append(Patch(
            facecolor=spec_color, edgecolor='none',
            label=rf"$\mathrm{{avgSpec}}_{{\text{{TPR-Range}}}} = {spec:.3f}$"
        ))

        leg = ax.legend(
            handles=handles,
            loc="lower right",
            bbox_to_anchor=(1.02, -0.02),
            fontsize=FS_LEGEND,
            framealpha=0.8,    
            facecolor="white", 
            edgecolor="none"
        )

        # cleanup
        for t in list(ax.texts):
            s = (t.get_text() or "").lower()
            if "inf" in s or "∞" in s or s.strip() == "0":
                t.remove()
        for coll in list(ax.collections):
            offsets = getattr(coll, "get_offsets", lambda: [])()
            if any((o[0] == 0 and o[1] == 0) for o in offsets):
                coll.remove()

        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)

        if TIGHT:
            fig.tight_layout()

        out_png = f"{base_path}.png"
        if len(tpr_groups) > 1:
            out_png = f"{base_path}_{i}.png"
            
        fig.savefig(out_png, dpi=DPI, bbox_inches="tight", transparent=TRANSPARENT)
        if save_pdf:
            fig.savefig(out_png.replace(".png", ".pdf"),
                        dpi=DPI, bbox_inches="tight", transparent=TRANSPARENT)

        if show:
            plt.show()
        plt.close(fig)

        outputs.append(out_png)

    return outputs


In [ ]:
# --------------------------- CONFIG ---------------------------
CLASS_NAME = "noninv" # <-- select model
FIG_DIR    = os.path.join("figures")
os.makedirs(FIG_DIR, exist_ok=True)

CACHE_DIR = "cache"

# TPR_GROUPS = [(0.75, 0.85)] # at mix and baseline
TPR_GROUPS = [(0.7, 0.8), (0.75, 0.85), (0.8, 0.9)] # at inv and noninv

# Minimal costs dict expected by deeproc if showOptimalROCpoints=True
COSTS_DICT = {
    "costsAreRates": True,
    # raw costs:
    "cFP": 1.0, "cFN": 1.0, "cTP": 0.0, "cTN": 0.0,
    # rate-named duplicates (einige Builds erwarten diese Keys):
    "cFPR": 1.0, "cFNR": 1.0, "cTPR": 0.0, "cTNR": 0.0,
}

In [ ]:
bundle = prepare_deeproc_class(CLASS_NAME, FIG_DIR, CACHE_DIR)
outs = plot_deeproc_from_bundle(
    bundle,
    tpr_groups=TPR_GROUPS,
    costs_dict=COSTS_DICT,
    out_dir=os.path.join("figures", CLASS_NAME),
    show=True,
    defaults={'fonts': dict(title=16, label=26, tick=22, legend=16), "lw_roc": 3.0, "lw_all": 1.5},
    save_pdf=True
)
print(outs)